
# Notebook with some usefull explorations of Whisper Model 

### What's that for?

This notebook contains few code experience that will be used further in project.  
I'm using it to explore possibale ways of inference and use.  
And ofcourse some experience with samples of target material.  
  
Whicn are:  
  
**The names of film takes extracted from the audio tracks that are voiced before the clapperboard strikes**

 Here will be some usefull thoughts and examples of local and cloud use



### Локальный запуск

>**примечание** 
>>   - модель скачивается и запускается локально исользуя механизм.  
>> 
>>   - при whisper-small это ооло 1000 gb и при запуске на cpu на моем i7 вполне работает.  
>> 
>>   - однако whisper-small значительно хуже более старших моделей - large - v3 вообще прекрасна - но значительно тяжелее. 
>>
>>   - если переставить флаг на gpu при наличии подходящей железки - будет в разы бодрей.  
>>
  

    
    
  
The Table of models actual on (29.05.2023):  

| Size | Parameters | English-only | Multilingual |
|------|------------|--------------|--------------|
| tiny | 39 M       | ✓            | ✓            |
| base | 74 M       | ✓            | ✓            |
| small | 244 M     | ✓            | ✓            |
| medium | 769 M    | ✓            | ✓            |
| large | 1550 M    | x            | ✓            |
| large-v2 | 1550 M | x            | ✓            |
| large-v3 | 1550 M | x            | ✓            |

https://huggingface.co/openai/whisper-large-v3


### Cloud inference 


Понадобиться:  
 - HK_TOKEN легко получить на HuggingFace
 - не успел разобраться что с Tiers 
 - есть несколько провайдеров на выбор + режим auto
```python

 import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    provider="hf-inference",
    api_key=os.environ["HF_TOKEN"],
)

output = client.automatic_speech_recognition("sample1.flac", model="openai/whisper-large-v3")

```

 - провайдеры: **fal-ai**, **replicate**, **auto**
 

In [1]:
import json
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

In [ ]:

# load model and processor
processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
forced_decoder_ids = processor.get_decoder_prompt_ids(language="french", task="transcribe")


In [ ]:
# load streaming dataset and read first audio sample
ds = load_dataset("distil-whisper/librispeech_long", "clean", split="validation")
ds = dataset[0]["audio"]
ds = ds.cast_column("audio", Audio(sampling_rate=16_000))
input_speech = next(iter(ds))["audio"]
input_features = processor(input_speech["array"], sampling_rate=input_speech["sampling_rate"], return_tensors="pt").input_features


In [ ]:

# generate token ids
predicted_ids = model.generate(input_features, forced_decoder_ids=forced_decoder_ids)
# decode token ids to text
transcription = processor.batch_decode(predicted_ids)
['<|startoftranscript|><|fr|><|transcribe|><|notimestamps|> Un vrai travail intéressant va enfin être mené sur ce sujet.<|endoftext|>']

transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
[' Un vrai travail intéressant va enfin être mené sur ce sujet.']

In [2]:

import os
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

device = "cpu"
torch_dtype = torch.float32  # на CPU только float32

model_id = "openai/whisper-small"

print("before model loading")

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True,
)
model.to(device)

# model.generation_config.forced_decoder_ids = None
# model.config.forced_decoder_ids = None

print("before processor loading")



before model loading


`torch_dtype` is deprecated! Use `dtype` instead!


before processor loading


In [4]:


processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
    
)

print("before inference")


# Передаёшь путь к своему файлу
result = pipe(
    "claps/тест.wav", 
    return_timestamps='True', 
    generate_kwargs={"task": "transcribe", "language": "ru"}
)
print(result["text"])

Device set to use cpu
`return_token_timestamps` is deprecated for WhisperFeatureExtractor and will be removed in Transformers v5. Use `return_attention_mask` instead, as the number of frames can be inferred from it.


before inference
 Привет-привет-привет! Бла-бла-бла! Ла-ля-ля! Как дела? Я не знаю, как наши дела! Бля-бля-бля!


In [5]:
import json
from rich import print as rprint

rprint(json.dumps(result, indent=4, ensure_ascii=False))

{
    "text": " Привет-привет-привет! Бла-бла-бла! Ла-ля-ля! Как дела? Я не знаю, как наши дела! Бля-бля-бля!",
    "chunks": [
        {
            "timestamp": [
                0.0,
                11.0
            ],
            "text": " Привет-привет-привет! Бла-бла-бла! Ла-ля-ля! Как дела? Я не знаю, как наши дела! 
Бля-бля-бля!"
        }
    ]
}

In [ ]:
import torch
import torchaudio
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

device = "cpu"
torch_dtype = torch.float32
model_id = "openai/whisper-small"

processor = AutoProcessor.from_pretrained(model_id)

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True,
).to(device)

model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None

waveform, sampling_rate = torchaudio.load("claps/Тест_шепот.wav")

# stereo -> mono
if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0, keepdim=True)

# 48000 -> 16000
if sampling_rate != 16000:
    waveform = torchaudio.functional.resample(
        waveform,
        orig_freq=sampling_rate,
        new_freq=16000,
    )
    sampling_rate = 16000

audio = waveform.squeeze().numpy()

input_features = processor(
    audio,
    sampling_rate=16000,
    return_tensors="pt",
).input_features.to(device)

predicted_ids = model.generate(
    input_features=input_features,
    language="ru",
    task="transcribe",
)

text = processor.batch_decode(
    predicted_ids,
    skip_special_tokens=True,
)[0]

print(text)

In [ ]:
import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    provider="fal-ai",
    # api_key=os.environ["HF_TOKEN"],
    api_key='HF_TOKEN_PLACEHOLDER',
)

output = client.automatic_speech_recognition(
    "claps/Тест_шепот.wav",
      model="openai/whisper-large-v3",
        extra_body={
            "language": "ru",
            "task": "translate",
            "timestamp_granularities": ["word"]
}
)

In [ ]:

print(output)

In [ ]:
from rich import print as rprint

In [ ]:
rprint(json.dumps(output, indent=4, ensure_ascii=False))

> Next cell helped me to convert my **.mva** files into **.wav**   
> Pretty easy  


In [ ]:
import subprocess

input_file = "claps/Тест_шепот.m4a"

subprocess.run(["m4a2wav", input_file], check=True)

wav_file = input_file.replace(".m4a", ".wav")